## NB-07 — Conversation Memory and Multi-Turn RAG

A single-turn assistant is not enough for real shoppers: after seeing earbuds
results, they ask "which of these support ANC?", then "what's the battery life
of the first one?", then "can I return it?". Each follow-up question depends on
the previous answer. This notebook adds conversation memory to CatalogueIQ using
LangChain's `ConversationalRetrievalChain`, tests a 5-turn shopper conversation,
and verifies that the system correctly resolves coreferences like "the first one
you mentioned" by reading the conversation history.

### Concepts Covered

- `ConversationalRetrievalChain` and how it differs from `RetrievalQA`
- `ConversationBufferWindowMemory`: what is stored, what is truncated
- Coreference resolution: how "it" and "the first one you mentioned" are resolved
- Why multi-turn context changes the embedding query: history-conditioned query rewriting
- Metadata filtering by category on product queries within a conversation
- Testing a 5-turn shopper conversation with the boAt Airdopes 141 as the anchor product
- Diagnosing memory failures: what goes wrong when the window is too small

In [ ]:
# ── SETUP ──────────────────────────────────────────────────────
import os
import json
import pickle
from pathlib import Path
from dotenv import load_dotenv
import faiss
from langchain.schema import Document
from langchain_anthropic import ChatAnthropic
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.vectorstores import FAISS as LangchainFAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate

load_dotenv()

MODEL = "claude-sonnet-4-6"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
INDEX_DIR = Path("data/index")
os.makedirs("output", exist_ok=True)

assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not found in .env"
print(f"Model: {MODEL}")


In [ ]:
# ── LOAD INDEX AND BUILD VECTORSTORE ─────────────────────────

index_path = INDEX_DIR / "faiss.index"
meta_path  = INDEX_DIR / "metadata.pkl"

if not index_path.exists():
    raise FileNotFoundError("Run NB-04 first to build the FAISS index.")

with open(meta_path, "rb") as f:
    all_docs = pickle.load(f)

hf_embedder = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = LangchainFAISS.from_documents(all_docs, hf_embedder)

print(f"Vectorstore ready: {len(all_docs)} documents indexed")


### Step 1 — Build the Conversational Retrieval Chain

`ConversationalRetrievalChain` has two stages:
1. **Condense question**: uses the conversation history to rewrite the current
   question into a standalone question (e.g. "which of these support ANC?" →
   "which wireless earbuds under ₹2000 support ANC?")
2. **Retrieve and generate**: retrieves context for the condensed question and
   generates a grounded answer.

`ConversationBufferWindowMemory` keeps the last `k=5` turns — enough for a
full 5-turn shopper conversation without hitting the context window limit.

In [ ]:
# ── CONVERSATION MEMORY ──────────────────────────────────────

# COST NOTE: ConversationalRetrievalChain makes TWO API calls per turn:
# one to condense the question (small) + one to generate the answer.
# 5 turns × 2 calls = 10 API calls ≈ $0.05–$0.10 total.

llm = ChatAnthropic(model=MODEL, temperature=0, max_tokens=1024)

# BufferWindowMemory keeps the last k=5 conversation exchanges
memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True,       # returns LangChain message objects (not plain text)
    output_key="answer",        # which chain output to store as the assistant turn
)

# System prompt for the conversational chain
# (applied to the final answer generation, not the condense step)
CONV_SYSTEM_PROMPT = """You are CatalogueIQ, the ShopSmart India product intelligence assistant.
You help shoppers find products, understand policies, and make purchasing decisions.

RULES:
- Never invent product specs. Only state facts from the retrieved context.
- Never invent policy rules. Always cite the source section.
- If a product is not in the context, say so explicitly.
- Include citations: [Source: <filename>, Product ID: <id> or Section: <heading>]
- Use ₹ for Indian Rupee amounts.
- When the shopper refers to a previous product ("the first one", "it", "that one"),
  use the conversation history to identify which product they mean.
- For follow-up questions about a specific product, focus your retrieval on that product.

Context from knowledge base:
{context}"""

conv_qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    memory=memory,
    return_source_documents=True,
    verbose=False,              # set True to see condense + answer prompts
    combine_docs_chain_kwargs={
        "prompt": ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(CONV_SYSTEM_PROMPT),
            HumanMessagePromptTemplate.from_template("{question}"),
        ])
    }
)

print("ConversationalRetrievalChain built.")
print(f"Memory: ConversationBufferWindowMemory(k=5)")


### Step 2 — 5-Turn Shopper Conversation

We simulate a realistic shopper session starting with a broad search and
progressively narrowing to specific product details and policy questions.
The critical test is Turn 3: "what is the battery life of the first one
you mentioned?" — the model must resolve "first one" from memory.

In [ ]:
# COST NOTE: 5 turns × 2 API calls each = 10 Claude API calls ≈ $0.05–$0.10

shopper_turns = [
    "Show me wireless earbuds under ₹2000",
    "Which of these support ANC?",
    "What is the battery life of the first one you mentioned?",
    "Can I return it if the sound quality is not good?",
    "Is the seller verified?",
]

conversation_log = []

print("─" * 65)
print("5-TURN SHOPPER CONVERSATION — ShopSmart India CatalogueIQ")
print("─" * 65)

for turn_num, user_message in enumerate(shopper_turns, 1):
    print(f"\n[Turn {turn_num}] Shopper: {user_message}")

    # COST NOTE: Two API calls per turn (condense question + generate answer)
    result = conv_qa_chain.invoke({"question": user_message})

    answer = result["answer"]
    sources = result.get("source_documents", [])

    print(f"[Turn {turn_num}] CatalogueIQ: {answer}")

    # Print which sources were retrieved (metadata only, not full content)
    source_refs = []
    for doc in sources:
        m = doc.metadata
        ref = m.get("product_id") or m.get("section_heading") or m.get("question_number", "?")
        source_refs.append(f"{m.get('source_file','?')}:{ref}")
    print(f"           [Sources: {', '.join(source_refs)}]")

    conversation_log.append({
        "turn": turn_num,
        "user": user_message,
        "assistant": answer,
        "sources": [d.metadata for d in sources],
    })

print("\n─" * 65)
print("\n✅ Check Turn 3: did the model correctly identify which product")
print("   'the first one you mentioned' refers to from the conversation history?")
print("   If not, check that memory.return_messages=True and output_key='answer'.")


### Step 3 — Inspect Conversation Memory Contents

After the 5-turn conversation, we inspect what is stored in memory to understand
what context the model has access to when resolving coreferences.

In [ ]:
# ── INSPECT MEMORY ────────────────────────────────────────────

print("── Conversation Memory Contents ─────────────────────────────")
memory_vars = memory.load_memory_variables({})
chat_history = memory_vars.get("chat_history", [])

print(f"Messages in memory: {len(chat_history)}")
for i, msg in enumerate(chat_history):
    role = "Human" if msg.type == "human" else "AI"
    content_preview = msg.content[:100].replace("\n", " ")
    print(f"  [{i+1}] {role}: {content_preview}…")

print("\n💡 Note: The 'first one you mentioned' in Turn 3 was resolved by")
print("   reading the AI message from Turn 1 in this memory buffer.")


In [ ]:
# ── SAVE CONVERSATION LOG ────────────────────────────────────

with open("output/nb07_conversation_log.json", "w", encoding="utf-8") as f:
    json.dump(conversation_log, f, indent=2, ensure_ascii=False)
print("Conversation log saved to output/nb07_conversation_log.json")


### Step 4 — Metadata Filtering on Product Queries

For product queries within a conversation, we can filter by category to improve
precision. This is especially useful when a shopper says "show me earbuds" —
we filter to `category=Electronics` before vector search.

In [ ]:
# ── METADATA FILTERING EXAMPLE ───────────────────────────────

# Create a filtered retriever for Electronics category only
electronics_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {"category": "Electronics"},   # metadata filter
    }
)

print("Testing category-filtered retrieval (Electronics only)...")
test_query = "wireless earbuds under 2000 rupees"
filtered_results = electronics_retriever.get_relevant_documents(test_query)

print(f"Query: '{test_query}'")
print(f"Results filtered to category=Electronics: {len(filtered_results)}")
for doc in filtered_results:
    m = doc.metadata
    print(f"  {m.get('product_id', '?')} | {m.get('product_name', '')[:40]} | "
          f"₹{m.get('price_inr', '?')}")

print("\n💡 In a production system, you would detect the query intent and")
print("   apply filters automatically: product query → filter by category,")
print("   policy query → filter by file_type='markdown'.")


In [ ]:
# ── EXPERIMENT ────────────────────────────────────────────────

# EXERCISE 1 — Memory window size
# Change memory = ConversationBufferWindowMemory(k=2) and replay the 5-turn conversation.
# At which turn does the model fail to resolve "the first one you mentioned"?
# This tells you the minimum memory window for a shopping assistance workflow.

# EXERCISE 2 — Cross-session memory (stretch)
# After the 5-turn conversation, save memory.chat_memory.messages to a JSON file.
# Create a new ConversationBufferWindowMemory and load the saved messages.
# Run Turn 6: "What was the product we discussed earlier with the longest battery life?"
# Does the restored memory allow correct resolution?

# EXERCISE 3 — Hinglish mid-conversation
# Replay the 5-turn conversation but change Turn 4 to:
# "Kya main isko return kar sakta hoon?" (Can I return this? — in Hindi)
# Does the model handle the language switch? Does retrieval still work?
# What does this tell you about multilingual support requirements?

print("Experiments ready. Modify memory k, add turns, or change languages above.")


### Key Takeaway

`ConversationalRetrievalChain` solves coreference resolution in multi-turn
shopping conversations by condensing each follow-up question into a standalone
query before retrieval. The memory buffer is the conversation context that makes
"the first one you mentioned" resolvable — without it, each turn is blind to
what came before. Memory window size is a real engineering tradeoff: too small
and coreferences break; too large and the condensed question prompt grows costly.

In **NB-08** you will bring all components together — ingestion, embedding,
expansion, re-ranking, memory, and generation — run the full RAGAS evaluation
against the golden test set from NB-02, identify the lowest-scoring query types,
make one targeted improvement, and measure the delta.